# 05. N-BEATS: Neural Basis Expansion for Time Series

## Background

N-BEATS (Neural Basis Expansion Analysis for Interpretable Time Series Forecasting, Oreshkin et al. 2019) was a landmark paper: the first deep learning model to consistently outperform classical statistical methods on large-scale benchmarks without any time series-specific inductive biases in its generic form.

The core idea: decompose the forecast into a sum of learned **basis functions**. Each block learns both a backcast (how well it can reconstruct the historical context) and a forecast (future prediction). The residual of the backcast is passed to the next block, creating a stack that progressively removes structure from the residuals.

The **interpretable** N-BEATS variant constrains the basis functions to be trend-specific (polynomial) and seasonality-specific (Fourier), making each block's contribution human-readable — you can see exactly what the trend block vs seasonality block contributes to the forecast.

## What You'll Learn

- N-BEATS architecture: blocks, stacks, backcast/forecast decomposition
- Generic vs interpretable variants
- Trend basis (polynomial) and seasonality basis (Fourier) expansion
- Why doubly residual connections improve gradient flow

## Why This Matters

N-BEATS demonstrated that pure MLPs — without recurrence or attention — could be world-class time series forecasters with the right architecture. This insight influenced subsequent work (N-HiTS, NBEATS-G) and provided theoretical grounding for basis function decomposition that transformer-based models implicitly learn.


## N-BEATS Block Implementation

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)

class NBEATSBlock(nn.Module):
    '''Single N-BEATS block: FC layers + backcast/forecast projections.'''

    def __init__(self, context_len: int, horizon: int,
                 n_theta_b: int, n_theta_f: int,
                 hidden_units: int = 256, n_layers: int = 4):
        super().__init__()
        self.context_len = context_len
        self.horizon = horizon

        # Shared FC stack
        layers = []
        in_size = context_len
        for _ in range(n_layers):
            layers += [nn.Linear(in_size, hidden_units), nn.ReLU()]
            in_size = hidden_units
        self.fc = nn.Sequential(*layers)

        # Basis expansion coefficients
        self.theta_b = nn.Linear(hidden_units, n_theta_b, bias=False)
        self.theta_f = nn.Linear(hidden_units, n_theta_f, bias=False)

    def forward(self, x: torch.Tensor) -> tuple:
        h = self.fc(x)
        theta_b = self.theta_b(h)
        theta_f = self.theta_f(h)
        return theta_b, theta_f

class GenericNBEATSBlock(NBEATSBlock):
    '''Generic block: learnable basis functions.'''

    def __init__(self, context_len: int, horizon: int,
                 n_theta: int = 32, **kwargs):
        super().__init__(context_len, horizon, n_theta, n_theta, **kwargs)
        # Backcast and forecast basis matrices are learned
        self.B_b = nn.Linear(n_theta, context_len, bias=False)
        self.B_f = nn.Linear(n_theta, horizon, bias=False)

    def forward(self, x: torch.Tensor) -> tuple:
        theta_b, theta_f = super().forward(x)
        backcast = self.B_b(theta_b)
        forecast = self.B_f(theta_f)
        return backcast, forecast

class TrendNBEATSBlock(NBEATSBlock):
    '''Interpretable trend block: polynomial basis.'''

    def __init__(self, context_len: int, horizon: int, degree: int = 3, **kwargs):
        super().__init__(context_len, horizon, degree+1, degree+1, **kwargs)
        self.degree = degree

        # Fixed polynomial basis matrices
        t_b = torch.linspace(0, 1, context_len).unsqueeze(0)  # (1, T)
        t_f = torch.linspace(0, 1, horizon).unsqueeze(0)
        # Vandermonde matrices: [t^0, t^1, ..., t^degree]
        B_b = torch.cat([t_b**i for i in range(degree+1)], dim=0).T  # (T, degree+1)
        B_f = torch.cat([t_f**i for i in range(degree+1)], dim=0).T
        self.register_buffer('B_b', B_b)
        self.register_buffer('B_f', B_f)

    def forward(self, x: torch.Tensor) -> tuple:
        theta_b, theta_f = super().forward(x)
        backcast = theta_b @ self.B_b.T  # (B, T)
        forecast = theta_f @ self.B_f.T  # (B, H)
        return backcast, forecast

class SeasonalNBEATSBlock(NBEATSBlock):
    '''Interpretable seasonality block: Fourier basis.'''

    def __init__(self, context_len: int, horizon: int,
                 n_harmonics: int = 10, **kwargs):
        n_theta = 2 * n_harmonics
        super().__init__(context_len, horizon, n_theta, n_theta, **kwargs)
        self.n_harmonics = n_harmonics

        def fourier_basis(length):
            t = torch.linspace(0, 1, length)
            freqs = torch.arange(1, n_harmonics + 1).float()
            cos_terms = torch.cos(2 * np.pi * freqs.unsqueeze(1) * t.unsqueeze(0))
            sin_terms = torch.sin(2 * np.pi * freqs.unsqueeze(1) * t.unsqueeze(0))
            return torch.cat([cos_terms, sin_terms], dim=0).T  # (length, 2*n_harmonics)

        self.register_buffer('B_b', fourier_basis(context_len))
        self.register_buffer('B_f', fourier_basis(horizon))

    def forward(self, x: torch.Tensor) -> tuple:
        theta_b, theta_f = super().forward(x)
        backcast = theta_b @ self.B_b.T
        forecast = theta_f @ self.B_f.T
        return backcast, forecast

# Test block shapes
CONTEXT, HORIZON = 52, 12
x_test = torch.randn(8, CONTEXT)

for BlockClass, name in [(GenericNBEATSBlock, 'Generic'),
                          (TrendNBEATSBlock, 'Trend (polynomial)'),
                          (SeasonalNBEATSBlock, 'Seasonal (Fourier)')]:
    if name == 'Generic':
        block = BlockClass(CONTEXT, HORIZON, n_theta=32, hidden_units=128)
    elif name.startswith('Trend'):
        block = BlockClass(CONTEXT, HORIZON, degree=3, hidden_units=128)
    else:
        block = BlockClass(CONTEXT, HORIZON, n_harmonics=8, hidden_units=128)
    backcast, forecast = block(x_test)
    print(f'{name}: backcast={tuple(backcast.shape)}, forecast={tuple(forecast.shape)}')


## Full N-BEATS Model with Stacks

In [ ]:
class NBEATS(nn.Module):
    def __init__(self, context_len: int, horizon: int,
                 n_generic_stacks: int = 2, n_blocks_per_stack: int = 3,
                 hidden_units: int = 256):
        super().__init__()
        self.context_len = context_len
        self.horizon = horizon

        # Stack 1: trend
        self.trend_blocks = nn.ModuleList([
            TrendNBEATSBlock(context_len, horizon, degree=3, hidden_units=hidden_units)
            for _ in range(n_blocks_per_stack)
        ])

        # Stack 2: seasonality
        self.seasonal_blocks = nn.ModuleList([
            SeasonalNBEATSBlock(context_len, horizon, n_harmonics=10, hidden_units=hidden_units)
            for _ in range(n_blocks_per_stack)
        ])

        # Stack 3+: generic
        self.generic_blocks = nn.ModuleList([
            GenericNBEATSBlock(context_len, horizon, n_theta=32, hidden_units=hidden_units)
            for _ in range(n_generic_stacks * n_blocks_per_stack)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        forecast = torch.zeros(x.size(0), self.horizon, device=x.device)

        for blocks in [self.trend_blocks, self.seasonal_blocks, self.generic_blocks]:
            for block in blocks:
                backcast, block_forecast = block(residual)
                residual = residual - backcast   # doubly residual
                forecast = forecast + block_forecast

        return forecast

nbeats = NBEATS(CONTEXT, HORIZON, n_generic_stacks=1, n_blocks_per_stack=2, hidden_units=128)
params = sum(p.numel() for p in nbeats.parameters())
print(f'N-BEATS: {params:,} parameters')

# Quick forward pass test
x = torch.randn(16, CONTEXT)
y = nbeats(x)
print(f'Input: {tuple(x.shape)}, Output: {tuple(y.shape)}')
print()
print('Architecture breakdown:')
print(f'  Trend blocks:    {len(nbeats.trend_blocks)} (polynomial basis, degree=3)')
print(f'  Seasonal blocks: {len(nbeats.seasonal_blocks)} (Fourier basis, 10 harmonics)')
print(f'  Generic blocks:  {len(nbeats.generic_blocks)} (learned basis)')
print(f'  Residual path:   doubly residual (backcast subtracted at each block)')
